# E1bs UNet-Strong Normal-Aware on Preprocessed BTXRD 224x224

Train the strong UNet recipe on preprocessed BTXRD with both tumor and normal cases. This run is designed to close Q3 by testing whether strong normal-aware training reduces false-positive masks on normal images.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/lehngoc/BTXRD-LViT.git"
BRANCH = "model/e1-unet-baseline"
REPO_ROOT = Path("/kaggle/working/BTXRD-LViT")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
else:
    print(f"Repo already exists: {REPO_ROOT}")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
import json
import platform
import zipfile

import pandas as pd
import torch
import yaml

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Change DATA_ROOT only if your Kaggle input path differs.
DATA_ROOT = Path("/kaggle/input/datasets/lehngoc/btxrd-preprocessed-dataset/btxrd-preprocessed")

def find_data_root() -> Path:
    target = Path("data/exports/btxrd_preprocessed/train.csv")
    if DATA_ROOT.exists() and (DATA_ROOT / target).exists():
        return DATA_ROOT
    for root in Path("/kaggle/input").glob("**"):
        if root.is_dir() and (root / target).exists():
            return root
    raise FileNotFoundError("Could not find BTXRD train.csv under /kaggle/input. Update DATA_ROOT.")

DATA_ROOT = find_data_root()
RUN_NAME = "E1bs_unet_strong_preprocessed_224_normal_aware"
OUTPUT_DIR = Path(f"/kaggle/working/experiments/{RUN_NAME}")
RUNTIME_CONFIG = Path(f"/kaggle/working/{RUN_NAME}.yaml")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("RUNTIME_CONFIG =", RUNTIME_CONFIG)

In [ ]:
required = [
    DATA_ROOT / "data/exports/btxrd_preprocessed/train.csv",
    DATA_ROOT / "data/exports/btxrd_preprocessed/val.csv",
    DATA_ROOT / "data/exports/btxrd_preprocessed/test.csv",
    DATA_ROOT / "data/processed/images_preprocessed",
    DATA_ROOT / "data/processed/masks_preprocessed",
]

for path in required:
    print(path, "->", path.exists())

missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))

In [ ]:
BASE_CONFIG = REPO_ROOT / "configs/train_unet_strong_normal_aware_preprocessed.yaml"
cfg = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))

cfg["data"]["root_dir"] = str(DATA_ROOT)
cfg["data"]["tumor_only"] = False
cfg["training"]["device"] = "cuda"
cfg["training"]["batch_size"] = 4
cfg["training"]["accumulation_steps"] = 2
cfg["training"]["num_workers"] = 2
cfg["training"]["epochs"] = 200
cfg["training"]["early_stopping_patience"] = 100
cfg["training"]["output_dir"] = str(OUTPUT_DIR)

RUNTIME_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(RUNTIME_CONFIG.read_text())

In [ ]:
for split in ["train", "val", "test"]:
    df = pd.read_csv(DATA_ROOT / f"data/exports/btxrd_preprocessed/{split}.csv")
    tumor = int(df["tumor"].astype(int).sum())
    normal = len(df) - tumor
    print(f"{split}: total={len(df)} tumor={tumor} normal={normal}")

## Smoke Test

In [ ]:
!python src/training/smoke_test_model_pipeline.py --config {RUNTIME_CONFIG} --samples-per-split 8

## Train E1bs

In [ ]:
!python src/training/train_unet.py --config {RUNTIME_CONFIG} --device cuda

## Evaluate Best Checkpoint

In [ ]:
best_ckpt = OUTPUT_DIR / "best.pt"
assert best_ckpt.exists(), best_ckpt

!python src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split val --device cuda --output {OUTPUT_DIR / 'val_metrics.json'}
!python src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split test --device cuda --output {OUTPUT_DIR / 'test_metrics.json'}

In [ ]:
for name in ["best_summary.json", "val_metrics.json", "test_metrics.json"]:
    path = OUTPUT_DIR / name
    print("\n===", name, "===")
    print(json.dumps(json.loads(path.read_text()), indent=2))

## Test Threshold Sweep

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
sweep = []
for thr in thresholds:
    out = OUTPUT_DIR / f"test_metrics_thr{int(thr * 100):02d}.json"
    !python src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split test --device cuda --threshold {thr} --output {out}
    metrics = json.loads(out.read_text())
    sweep.append({
        "threshold": thr,
        "tumor_dice": metrics["tumor_dice"],
        "tumor_iou": metrics["tumor_iou"],
        "tumor_precision": metrics["tumor_precision"],
        "tumor_recall": metrics["tumor_recall"],
        "normal_count": metrics["normal_count"],
        "normal_pred_area_ratio": metrics["normal_pred_area_ratio"],
        "normal_fp_image_rate": metrics["normal_fp_image_rate"],
    })

(OUTPUT_DIR / "test_threshold_sweep_metrics.json").write_text(json.dumps(sweep, indent=2), encoding="utf-8")
display(pd.DataFrame(sweep))

## Visualize Normal False Positives

In [ ]:
for thr in [0.5, 0.6, 0.7]:
    out_dir = OUTPUT_DIR / "visual_checks" / f"thr{int(thr * 100):02d}"
    !python src/training/visualize_unet_predictions.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split test --device cuda --threshold {thr} --output-dir {out_dir} --max-tumor 0 --max-normal 24

## Package Outputs

In [ ]:
metrics_zip = Path("/kaggle/working/E1bs_unet_strong_normal_aware_metrics_only.zip")
wanted = [
    "history.csv",
    "best_summary.json",
    "val_metrics.json",
    "test_metrics.json",
    "test_threshold_sweep_metrics.json",
    "config.json",
]
with zipfile.ZipFile(metrics_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in wanted:
        path = OUTPUT_DIR / name
        if path.exists():
            z.write(path, arcname=name)
    for path in sorted(OUTPUT_DIR.glob("test_metrics_thr*.json")):
        z.write(path, arcname=path.name)

full_zip = Path("/kaggle/working/E1bs_unet_strong_normal_aware_full_artifacts.zip")
with zipfile.ZipFile(full_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            z.write(path, arcname=str(path.relative_to(OUTPUT_DIR)))

print("metrics zip:", metrics_zip, metrics_zip.stat().st_size / 1024 / 1024, "MB")
print("full zip:", full_zip, full_zip.stat().st_size / 1024 / 1024, "MB")